# 연습 문제 1-6, 1-7 풀이: 연산 그래프, 기울기, topk 마스크

1-2절 연습 문제 1-6, 1-7의 풀이 예시다. 정답이 하나뿐인 문제가 아니므로 다른 방법으로도 구현할 수 있다.

- 본문 예제 코드는 `code_examples/ch01/` 아래 예제 노트북을 참고한다.

In [1]:
# torch 라이브러리 임포트
import torch

## 연습 문제 1-6

> [코드 1-25]에서 1일 총비용 계산의 연산 그래프를 종이에 그려 보자.

**연산 그래프를 그리려면**
- 계산 과정을 순서대로 나눈다: `get_prices()` → `get_cost()`
- 텐서는 노드, 연산은 노드를 잇는 화살표로 그린다
- 마지막 결과(`cost_1day`)가 스칼라여야 `backward()`를 호출할 수 있다

In [2]:
# [코드 1-25]의 설정: 재료 사용량, 판매량, 재료별 단위가격(자동 미분 대상)
ingredients = torch.tensor([[10, 10, 0, 8, 2], [0, 15, 10, 5, 1]])           # 카페라테, 코코아의 재료 사용량
sales_1day = torch.tensor([20, 10])                                          # 음료별 판매량
unit_costs = torch.tensor([100., 100., 100., 100., 100.], requires_grad=True)  # 재료별 단위가격

def get_prices(unit_costs, ingredients):
    return (unit_costs * ingredients).sum(dim=1)     # 음료별 생산 비용

def get_cost(prices, sales):
    return (prices * sales).sum()                    # 1일 총비용

**1일 총비용 계산의 연산 그래프**

```
unit_costs (5,)       ingredients (2, 5)
        \                 /
         ( × )  요소별 곱, 브로드캐스팅 → (2, 5)
              |
         ( ∑ dim=1 )  재료 축을 따라 합 → prices (2,)      sales_1day (2,)
              \                                              /
                         ( × )  요소별 곱 → (2,)
                              |
                         ( ∑ )  전체 합
                              |
                      cost_1day (스칼라)
```
- 위에서 아래로 가는 화살표 방향이 순전파이고, 기울기는 반대 방향으로 거슬러 올라가며 계산된다

In [3]:
prices = get_prices(unit_costs, ingredients)     # 순전파 1단계: 음료별 생산 비용 (2,)
cost_1day = get_cost(prices, sales_1day)         # 순전파 2단계: 1일 총비용 (스칼라)
cost_1day.backward()                             # 역전파
print(f'prices: {prices}')                       # grad_fn이 연산 그래프의 흔적
print(f'cost_1day: {cost_1day}')
print(f'unit_costs.grad: {unit_costs.grad}')

prices: tensor([3000., 3100.], grad_fn=<SumBackward1>)
cost_1day: 91000.0
unit_costs.grad: tensor([200., 350., 100., 210.,  50.])


**그린 그래프가 맞는지 기울기로 확인하기**
- 종이에 그린 그래프는 답을 맞춰 볼 방법이 없다
- 그린 그래프로 `unit_costs.grad`의 값을 설명할 수 있으면 맞게 그린 것
- 우유(두 번째 재료): 카페라테 10 × 20잔 + 코코아 15 × 10잔 = 350

In [4]:
# 우유(두 번째 재료)의 기울기를 손으로 계산: 카페라테 10 x 20잔 + 코코아 15 x 10잔
print(f'우유: {10 * 20 + 15 * 10}')
# 모든 재료: 재료 사용량 x 판매량을 음료 방향으로 합함(1-1절의 브로드캐스팅)
by_hand = (ingredients * sales_1day.unsqueeze(1)).sum(dim=0)
print(f'손으로 계산: {by_hand}')
print(f'자동 미분 결과와 같은가: {torch.equal(by_hand.float(), unit_costs.grad)}')

우유: 350
손으로 계산: tensor([200, 350, 100, 210,  50])
자동 미분 결과와 같은가: True


## 연습 문제 1-7 [도전 문제]

> 카페 주인과 재료 공급업자는 모든 재료의 단위가격을 한 번에 낮추기 어렵다고 판단했다.
> 대신 한 번 최적화할 때마다 영향력이 가장 큰 두 재료의 단위가격만 낮추기로 했다.
> 이 방식으로 20회 반복해 최적화하는 과정을 구현해 보자.
>
> 힌트: 각 반복에서 기울기를 구한 뒤 다음 반복 전에 이전 기울기를 초기화하므로,
> 그 사이에는 `grad` 속성의 값을 자유롭게 바꿔도 된다.

**영향력이 가장 큰 두 재료만 고르려면**
- `grad.topk(2).indices`로 기울기가 큰 두 재료의 위치를 찾는다
- 그 두 자리만 1, 나머지는 0인 텐서로 마스크를 만들어 기울기에 곱한다
- 선택되지 않은 재료는 기울기가 0이 되어 단위가격이 그대로 유지된다

In [5]:
ITERATION, DISCOUNT_RATE, TOP_K = 20, 0.01, 2
unit_costs = torch.tensor([100., 100., 100., 100., 100.], requires_grad=True)

for i in range(ITERATION):
    unit_costs.grad = None
    get_cost(get_prices(unit_costs, ingredients), sales_1day).backward()
    with torch.no_grad():
        top_indices = unit_costs.grad.topk(TOP_K).indices   # 가장 큰 두 위치
        mask = torch.zeros_like(unit_costs.grad)
        mask[top_indices] = 1.                              # 그 자리만 1
        unit_costs -= DISCOUNT_RATE * unit_costs.grad * mask
    if i in (0, 1, ITERATION - 1):
        prices_now = [round(c, 1) for c in unit_costs.tolist()]
        print(f'{i + 1:2d}번째: 선택 {top_indices.tolist()}, 단위가격 {prices_now}')

 1번째: 선택 [1, 3], 단위가격 [100.0, 96.5, 100.0, 97.9, 100.0]
 2번째: 선택 [1, 3], 단위가격 [100.0, 93.0, 100.0, 95.8, 100.0]
20번째: 선택 [1, 3], 단위가격 [100.0, 30.0, 100.0, 58.0, 100.0]


**결과: 20번 내내 같은 재료가 선택된다**
- 우유(1번)와 전기(3번)만 계속 선택되고, 나머지 세 재료는 100원 그대로
- 총비용이 단위가격의 일차식이라 기울기가 늘 같기 때문(맞게 구현한 것!)

In [6]:
# 20회 최적화한 뒤의 기울기를 다시 구해 처음과 비교
unit_costs.grad = None
get_cost(get_prices(unit_costs, ingredients), sales_1day).backward()
print(f'20회 최적화 뒤의 기울기: {unit_costs.grad.tolist()}')

20회 최적화 뒤의 기울기: [200.0, 350.0, 100.0, 210.0, 50.0]


참고: **기울기가 달라지는 모델이라면?**
- 1-3절의 회귀 분석 모델은 파라미터 값에 따라 기울기가 달라진다
- 그런 모델에서는 반복할 때마다 선택되는 파라미터도 달라질 수 있다

**직접 해 보기: 갱신을 `-=` 대신 뺄셈 대입으로 쓰면?**
- 어디에서 예외가 발생할까? 왜 발생할까?
- 실행하기 전에 먼저 예측해 보자

In [ ]:
# 갱신식을 -= 대신 뺄셈 대입으로 바꾼 코드
unit_costs = torch.tensor([100., 100., 100., 100., 100.], requires_grad=True)
for i in range(2): 
    unit_costs.grad = None
    get_cost(get_prices(unit_costs, ingredients), sales_1day).backward()
    with torch.no_grad():
        unit_costs = unit_costs - DISCOUNT_RATE * unit_costs.grad

RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

**원인**
- 뺄셈은 결과를 새 텐서로 만들고, 새 텐서는 `requires_grad`가 `False`
- 그래서 두 번째 반복의 계산에는 연산 그래프가 없어 `backward()`를 호출할 수 없다
- 자동 미분 대상 텐서는 `-=` 같은 제자리 연산으로 값만 바꿔야 한다